In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

TRAINING_DEVICE = "cuda:0"
model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [4]:
import os
from concurrent.futures import ThreadPoolExecutor

import cloudpickle
import numpy as np

from transformers_sae.ops import (
    ensure_directory,
    find_checkpoint_after,
    load_checkpoint,
    save_validations,
)
from transformers_sae.training import tune_activation_thresholds
from transformers_sae.validation import run_validations


def load_saes(checkpoint_dir: str, tokens: int):
    saes = {}

    def load_layer_checkpoint(layer):
        checkpoint, actual_tokens = find_checkpoint_after(checkpoint_dir, layer, tokens)
        try:
            assert actual_tokens >= tokens
            sae = load_checkpoint(checkpoint).sae
            sae.eval()
            sae.onload()
            print(f"Loaded checkpoint for layer {layer}")
            return layer, sae
        except Exception:
            print(f"No checkpoint found for layer {layer}")
            raise
            return layer, None

    # Load the latest checkpoints for each layer in parallel
    with ThreadPoolExecutor() as executor:
        results = executor.map(
            load_layer_checkpoint, range(model.num_layers - 1, -1, -1)
        )
        for layer, sae in results:
            if sae is not None:
                saes[layer] = sae

    return saes


METHOD_NAME = "next_layer_interaction"
CHECKPOINT_DIR = f"{os.getenv('HF_BUCKET_LOCAL')}/gemma_2_2b/{METHOD_NAME}"

for num_tokens in range(int(1e7), int(1e8) + int(1e7), int(1e7)):
    results_path = (
        f"{os.getenv('HF_BUCKET_LOCAL')}/training_validation/gemma_2_2b/{METHOD_NAME}/"
    )
    ensure_directory(results_path)
    print(f"Validation for {num_tokens}")
    saes = load_saes(CHECKPOINT_DIR, num_tokens)
    tune_activation_thresholds(
        model,
        tokenizer,
        saes,
        training_dataset,
        256,
        1,
        int(1e6),
        offload_after_training=False,
    )
    new_thresholds = {
        layer: tuple(a.threshold.item() for a in sae.encoder.activation)
        for layer, sae in saes.items()
    }
    with open(f"{results_path}/{num_tokens}.activation_thresholds", "wb") as f:
        cloudpickle.dump(new_thresholds, f)
    validations = run_validations(
        model,
        tokenizer,
        saes,
        validation_dataset,
        256,
        1,
        int(1e6),
        start_layer=0,
        offload=False,
    )
    print(
        f"{METHOD_NAME} {num_tokens} tokens KL: ",
        np.exp(
            np.mean(
                np.log(
                    np.clip(
                        validations.layer_results[model.num_layers].kl,
                        min=1e-9,
                    )
                )
            )
        ).item(),
    )
    save_validations({0: validations}, f"{results_path}/{num_tokens}.validations")


Validation for 10000000
Loaded checkpoint for layer 18
Loaded checkpoint for layer 19
Loaded checkpoint for layer 12
Loaded checkpoint for layer 21
Loaded checkpoint for layer 17
Loaded checkpoint for layer 25
Loaded checkpoint for layer 14
Loaded checkpoint for layer 7
Loaded checkpoint for layer 0
Loaded checkpoint for layer 22
Loaded checkpoint for layer 16
Loaded checkpoint for layer 10
Loaded checkpoint for layer 8
Loaded checkpoint for layer 9
Loaded checkpoint for layer 3
Loaded checkpoint for layer 15
Loaded checkpoint for layer 13
Loaded checkpoint for layer 1
Loaded checkpoint for layer 5
Loaded checkpoint for layer 20
Loaded checkpoint for layer 6
Loaded checkpoint for layer 23
Loaded checkpoint for layer 24
Loaded checkpoint for layer 11
Loaded checkpoint for layer 4
Loaded checkpoint for layer 2


Tuning BatchTopK thresholds for replacement model

  0%|          | 0/1000000 [00:00<?, ?it/s]

Running SAE evals:   0%|          | 0/1000000 [00:00<?, ?it/s]

next_layer_interaction 10000000 tokens KL:  1.8334227800369263
Validation for 20000000
Loaded checkpoint for layer 5
Loaded checkpoint for layer 3
Loaded checkpoint for layer 7
Loaded checkpoint for layer 6
Loaded checkpoint for layer 13
Loaded checkpoint for layer 10
Loaded checkpoint for layer 18
Loaded checkpoint for layer 11
Loaded checkpoint for layer 21
Loaded checkpoint for layer 0
Loaded checkpoint for layer 2
Loaded checkpoint for layer 25
Loaded checkpoint for layer 23
Loaded checkpoint for layer 14
Loaded checkpoint for layer 12
Loaded checkpoint for layer 22
Loaded checkpoint for layer 16
Loaded checkpoint for layer 20
Loaded checkpoint for layer 1
Loaded checkpoint for layer 9
Loaded checkpoint for layer 15
Loaded checkpoint for layer 19
Loaded checkpoint for layer 24
Loaded checkpoint for layer 17
Loaded checkpoint for layer 8
Loaded checkpoint for layer 4


...

next_layer_interaction 90000000 tokens KL:  1.4470878839492798
Validation for 100000000
Loaded checkpoint for layer 4
Loaded checkpoint for layer 22
Loaded checkpoint for layer 23
Loaded checkpoint for layer 10
Loaded checkpoint for layer 25
Loaded checkpoint for layer 1
Loaded checkpoint for layer 21
Loaded checkpoint for layer 20
Loaded checkpoint for layer 19
Loaded checkpoint for layer 14
Loaded checkpoint for layer 15
Loaded checkpoint for layer 6
Loaded checkpoint for layer 16
Loaded checkpoint for layer 11
Loaded checkpoint for layer 3
Loaded checkpoint for layer 18
Loaded checkpoint for layer 12
Loaded checkpoint for layer 8
Loaded checkpoint for layer 7
Loaded checkpoint for layer 0
Loaded checkpoint for layer 24
Loaded checkpoint for layer 2
Loaded checkpoint for layer 9
Loaded checkpoint for layer 5
Loaded checkpoint for layer 13
Loaded checkpoint for layer 17


Tuning BatchTopK thresholds for replacement model

  0%|          | 0/1000000 [00:00<?, ?it/s]

Running SAE evals:   0%|          | 0/1000000 [00:00<?, ?it/s]

next_layer_interaction 100000000 tokens KL:  1.2780325412750244
